In [ ]:
import resource
from functools import partial
from time import time

import tensorflow_datasets
from architecture import (
    BOUNDS_NORMA,
    denormalised_preds,
    load_state,
    preprocessing_ellips,
    preprocessing_shear,
)
from flax.core.frozen_dict import freeze
from jax import numpy, random
from pandas import DataFrame, read_csv
from tqdm import tqdm
from utils import (
    compute_mode,
    get_arg,
    get_resnet_blocks,
    get_resnet_blocks2,
    params_for_multi,
    params_for_uni,
    sampling_pdf_multi,
    sampling_pdf_uni,
    search_folder,
    simplify,
)
from utils_plot import (
    channels_layers,
    combined_plots,
    histos,
    loss_optimiser,
    pdf_multi,
    pdf_uni,
    plot_cams,
    plot_examples,
    predictions,
    shape_network,
)


def delete_lists():
    for name in list(globals()):
        if name.startswith("list_"):
            del globals()[name]


resource.setrlimit(resource.RLIMIT_NOFILE, (10000, 10000))

FOLDER_RESULTS = search_folder(name="RESULTS")
print("OPTIONS : \n")
folders = sorted([folder for folder in FOLDER_RESULTS.iterdir() if folder.is_dir()])
for folder_name in folders:
    print(folder_name.name)
del folders, folder_name

## choice

In [ ]:
CHOICE = (
    "Net-10_batch-64_steps-40000_db-2.0.1_pred-ellips_cpt-6_prelayer-32_layer-2222_kernel-3_q-uni"  # noqa
)
PERCENTAGE = 98

## load model

In [3]:
STYLE = get_arg(CHOICE, "q")
DB = get_arg(CHOICE, "db")
PREDICT = get_arg(CHOICE, "pred")

FOLDER_RESULTS = search_folder(name="RESULTS") / CHOICE
MODEL, STATE = load_state(FOLDER_RESULTS)

FOLDER_DATABASE = search_folder(name="pisco_euclid") / DB

In [4]:
stats = read_csv(FOLDER_DATABASE / "statistics.csv")
std_images = numpy.float32(stats["std"].iloc[0])

if PREDICT == "shear":
    min = numpy.float32(stats["min_shears"].iloc[0])
    max = numpy.float32(stats["max_shears"].iloc[0])
    PROCESS_FUNCTION = partial(preprocessing_shear, std_images=std_images, old_min=min, old_max=max)

elif PREDICT == "ellips":
    min = numpy.float32(stats["min_ellips"].iloc[0])
    max = numpy.float32(stats["max_ellips"].iloc[0])
    PROCESS_FUNCTION = partial(
        preprocessing_ellips, std_images=std_images, old_min=min, old_max=max
    )

## loader

In [ ]:
test_dataset = tensorflow_datasets.load(
    f"pisco_euclid:{DB}",
    split=f"train[{PERCENTAGE}%:]",
    data_dir=search_folder(name="DATASETS"),
    shuffle_files=False,
)
print("total NB_ELEMENTS", TOTAL_ELEMENTS := test_dataset.cardinality().numpy())

all_params_dataset = test_dataset.map(
    lambda x: {
        "avg_e": x["avg_e"],
        "avg_g": x["avg_g"],
        "image": x["image"],
        "gal_params": x["gal_params"],
    }
)

all_params = list(all_params_dataset.as_numpy_iterator())
ellips = numpy.array([item["avg_e"] for item in all_params])
shears = numpy.array([item["avg_g"] for item in all_params])
images = numpy.array([item["image"] for item in all_params])
gal_params = numpy.array([item["gal_params"] for item in all_params])

hypercube_images = DataFrame(
    numpy.hstack([shears, ellips]), columns=["avg_g1", "avg_g2", "avg_e1", "avg_e2"]
)

BATCHES = 50
test_loader = test_dataset.batch(BATCHES).map(PROCESS_FUNCTION).as_numpy_iterator()
del test_dataset, all_params, all_params_dataset

## inference

In [ ]:
if STYLE == "multiscalar":
    print(STYLE)
    modes = []
    for batch in tqdm(test_loader, total=int(TOTAL_ELEMENTS / BATCHES)):
        variables = {"params": STATE["params"], "batch_stats": STATE["batch_stats"]}
        normalised_images = numpy.nan_to_num(batch[0])
        modes.append(MODEL.apply(variables, normalised_images))
    modes = numpy.concatenate(modes)


elif STYLE == "multi":
    print(STYLE)
    list_weights, list_means, list_stddevs = [], [], []
    list_pdfs, list_modes, list_xy_values = [], [], []
    NB_POINTS = 300
    x_values = numpy.linspace(BOUNDS_NORMA[0], BOUNDS_NORMA[1], NB_POINTS)
    xy_values = numpy.stack([x_values, x_values], axis=0)
    for batch in tqdm(test_loader, total=int(TOTAL_ELEMENTS / BATCHES)):
        variables = {
            "params": STATE["params"],
            "batch_stats": STATE["batch_stats"],
            "perturbations": STATE["perturbations"],
        }
        normalised_images = numpy.nan_to_num(batch[0])
        q = MODEL.apply(variables, normalised_images)

        weights, means, stddevs = params_for_multi(q)
        list_weights.append(weights)
        list_means.append(means)
        list_stddevs.append(stddevs)

        pdfs, modes = sampling_pdf_multi(weights, means, stddevs, xy_values)
        list_modes.append(modes)
        list_pdfs.append(pdfs)

    weights = numpy.concatenate(list_weights)
    means = numpy.concatenate(list_means)
    stddevs = numpy.concatenate(list_stddevs)
    modes = numpy.concatenate(list_modes)
    pdfs = numpy.concatenate(list_pdfs)

    del q, x_values, NB_POINTS


elif STYLE == "uni":
    print(STYLE)
    STEP = 1e-5
    nb_of_x_values = int((max - min) / STEP)
    x_values = numpy.linspace(BOUNDS_NORMA[0], BOUNDS_NORMA[1], nb_of_x_values)

    list_weights, list_means, list_stddevs = [], [], []
    list_modes_g1, list_modes_g2, list_pdfs = [], [], []

    # computing args and modes
    for batch in tqdm(test_loader, total=int(TOTAL_ELEMENTS / BATCHES)):
        variables = {
            "params": STATE["params"],
            "batch_stats": STATE["batch_stats"],
            "perturbations": STATE["perturbations"],
        }
        normalised_images = numpy.nan_to_num(batch[0])
        q1, q2 = MODEL.apply(variables, normalised_images)
        weights, means, stddevs = params_for_uni(q1, q2)

        # Keep results
        list_weights.append(weights)
        list_means.append(means)
        list_stddevs.append(stddevs)

        # Get pdfs from direct network results
        pdf1 = sampling_pdf_uni(weights[:, 0, :], means[:, 0, :], stddevs[:, 0, :], x_values)
        pdf2 = sampling_pdf_uni(weights[:, 1, :], means[:, 1, :], stddevs[:, 1, :], x_values)
        list_pdfs.append(numpy.stack([pdf1, pdf2], axis=1))

        # Get modes from sampling network distribution
        for i in range(weights.shape[0]):
            mode_g1 = compute_mode(pdf1[i], x_values)
            mode_g2 = compute_mode(pdf2[i], x_values)
            list_modes_g1.append(mode_g1)
            list_modes_g2.append(mode_g2)

    weights = numpy.concatenate(list_weights)
    means = numpy.concatenate(list_means)
    stddevs = numpy.concatenate(list_stddevs)
    pdfs = numpy.concatenate(list_pdfs)
    modes = numpy.stack((numpy.array(list_modes_g1), numpy.array(list_modes_g2))).T

    del q1, q2, nb_of_x_values, STEP, pdf1, pdf2, mode_g1, mode_g2


delete_lists()
del batch, variables, test_loader, normalised_images

## postprocess

In [7]:
if STYLE == "multiscalar":
    modes = denormalised_preds(modes, min, max)
elif STYLE == "multi":
    means = denormalised_preds(means, min, max)
    modes = denormalised_preds(modes, min, max)
    xy_values = denormalised_preds(xy_values, min, max)
elif STYLE == "uni":
    means = denormalised_preds(means, min, max)
    modes = denormalised_preds(modes, min, max)
    x_values = denormalised_preds(x_values, min, max)

In [8]:
name_columns = hypercube_images.columns
hypercube_images = hypercube_images.to_numpy()
for i in range(2):
    means = means[numpy.abs(shears[:, i]) <= 0.05]
    weights = weights[numpy.abs(shears[:, i]) <= 0.05]
    modes = modes[numpy.abs(shears[:, i]) <= 0.05]
    pdfs = pdfs[numpy.abs(shears[:, i]) <= 0.05]
    hypercube_images = hypercube_images[numpy.abs(shears[:, i]) <= 0.05]
    images = images[numpy.abs(shears[:, i]) <= 0.05]
    shears = shears[numpy.abs(shears[:, i]) <= 0.05]
hypercube_images = DataFrame(hypercube_images, columns=name_columns)
del i

## losses

In [ ]:
first_step = 0
last_step = 1000

data = numpy.load(FOLDER_RESULTS / "training_data.npz")
loss_optimiser(
    data["train_loss"],
    data["val_loss"],
    FOLDER_RESULTS,
    data["time"],
    data["steps"],
    first_step,
    last_step,
    # ylims=[0, 1],
)

## results

In [ ]:
if PREDICT == "ellips":
    bornes_res = 3e-1
    symbol = "epsilon"
elif PREDICT == "shear":
    bornes_res = 1.0e-1
    symbol = "gamma"
predictions(modes, shears, FOLDER_RESULTS / "results", symbol=symbol, bornes_res=bornes_res)
del bornes_res

## errors

In [ ]:
nb_of_bins = 40
combined_plots(modes, shears, nb_of_bins, FOLDER_RESULTS)
del nb_of_bins

## histo

In [ ]:
if STYLE != "multiscalar":
    if STYLE == "multi":
        row_max_indices = numpy.argmax(weights, axis=1)
        mask = numpy.zeros_like(weights, dtype=bool)
        mask = mask.at[numpy.arange(weights.shape[0]), row_max_indices].set(True)

        masked_values_g1 = means[:, 0, :][mask]
        masked_values_g2 = means[:, 1, :][mask]

    if STYLE == "uni":
        row_max_indices = numpy.argmax(weights[:, 0, :], axis=1)
        mask = numpy.zeros_like(weights[:, 0, :], dtype=bool)
        mask = mask.at[numpy.arange(weights.shape[0]), row_max_indices].set(True)
        masked_values_g1 = means[:, 0, :][mask]

        row_max_indices = numpy.argmax(weights[:, 1, :], axis=1)
        mask = numpy.zeros_like(weights[:, 1, :], dtype=bool)
        mask = mask.at[numpy.arange(weights.shape[0]), row_max_indices].set(True)
        masked_values_g2 = means[:, 1, :][mask]

    nb_of_bins = 40
    histos(weights, means, masked_values_g1, masked_values_g2, nb_of_bins, STYLE, FOLDER_RESULTS)
    del nb_of_bins

## pdf

In [ ]:
CHOICES = [75]
# zoom = 2e-2
zoom = 1e-1

for CHOIX in CHOICES:
    if STYLE == "multi":
        pdf_multi(
            pdfs[CHOIX],
            modes[CHOIX],
            shears[CHOIX],
            xy_values,
            CHOIX,
            FOLDER_RESULTS,
        )

    elif STYLE == "uni":
        pdf_uni(
            x_values=x_values,
            pdf=pdfs[CHOIX],
            mode=modes[CHOIX],
            truth=shears[CHOIX],
            choix=CHOIX,
            zoom=zoom,
            directory=FOLDER_RESULTS,
            mean=means[CHOIX],
        )

del CHOICES, CHOIX, zoom

## cam

In [ ]:
LIST_NUMBERS = [100, 300]
compress = True

for NUMBER in LIST_NUMBERS:
    TEST_IMAGE = images[NUMBER].reshape(1, 352, 352, 1)
    TEST_SHEAR = shears[NUMBER]

    variables = {
        "params": STATE["params"],
        "batch_stats": STATE["batch_stats"],
        "perturbations": STATE["perturbations"],
    }
    _, state_intermediates = MODEL.apply(variables=variables, x=TEST_IMAGE, mutable="intermediates")

    try:
        all_conv_layers = get_resnet_blocks2(state_intermediates["intermediates"]["Encoder"])
    except KeyError:
        all_conv_layers = get_resnet_blocks(state_intermediates["intermediates"])
    final_conv_layer = state_intermediates["intermediates"]["final_conv_layer"][0][0]
    perturbations = freeze({"conv_1": final_conv_layer})

    pooling_cam = numpy.mean(final_conv_layer, axis=(0, 1))
    cam = numpy.dot(final_conv_layer, pooling_cam)
    # cam = numpy.sum(final_conv_layer * pooling_cam, axis=2)

    plot_cams(TEST_IMAGE.reshape(352, 352), NUMBER, cam, FOLDER_RESULTS, compress=True, name="cam")
    channels_layers(all_conv_layers, NUMBER, which=1, compress=compress, directory=FOLDER_RESULTS)
    channels_layers(all_conv_layers, NUMBER, which=2, compress=compress, directory=FOLDER_RESULTS)
    try:
        channels_layers(
            all_conv_layers, NUMBER, which=3, compress=compress, directory=FOLDER_RESULTS
        )
    except IndexError:
        pass

## examples

In [15]:
compress = True
nb_plots = 2
random_indices = random.permutation(random.PRNGKey(int(time())), modes.shape[0])[:nb_plots]

In [ ]:
name = "_" + "_".join(map(str, random_indices)) + "_"
plot_examples(
    modes[random_indices],
    images[random_indices],
    hypercube_images.iloc[random_indices],
    compress,
    name,
    FOLDER_RESULTS,
)
del random_indices, nb_plots

## gradcam

In [17]:
# from jax import grad

# def loss_multiscalar_gradcam(params, batch_stats, perturbations, x, y):
#     preds = MODEL.apply(
#         {"params": params, "batch_stats": batch_stats, "perturbations": perturbations}, x
#     )
#     return numpy.mean(numpy.sum((preds - y) ** 2, axis=-1))


# def loss_multi_gradcam(params, batch_stats, perturbations, x, y):
#     q = MODEL.apply(
#         {"params": params, "batch_stats": batch_stats, "perturbations": perturbations}, x
#     )
#     return numpy.mean(-q.log_prob(y))


# def loss_uni_gradcam(params, batch_stats, perturbations, x, y):
#     q1, q2 = MODEL.apply(
#         {"params": params, "batch_stats": batch_stats, "perturbations": perturbations}, x
#     )
#     return numpy.mean(-q1.log_prob(y[:, 0]) - q2.log_prob(y[:, 1]))


# if STYLE == "multiscalar":
#     grad_fn = grad(loss_multiscalar_gradcam, argnums=2)
# elif STYLE == "multi":
#     grad_fn = grad(loss_multi_gradcam, argnums=2)
# elif STYLE == "uni":
#     grad_fn = grad(loss_uni_gradcam, argnums=2)
# final_conv_grads = grad_fn(
#     STATE["params"], STATE["batch_stats"], STATE["perturbations"], TEST_IMAGE, TEST_SHEAR[None, ...]#noqa
# )
# final_conv_grads = final_conv_grads["conv_1"][0]
# pooling_gradcam = numpy.mean(final_conv_grads, axis=(0, 1))
# # gradcam = numpy.sum(
# #     final_conv_grads * pooling_gradcam, axis=2
# # )
# gradcam = numpy.dot(final_conv_grads, pooling_gradcam)
# plot_cams(TEST_IMAGE.reshape(352, 352), gradcam, FOLDER_RESULTS, compress=True, name="gradcam")

## shapes

In [ ]:
MODEL_CONFIG = simplify(STATE["model_config"])
MODEL_CONFIG

if MODEL_CONFIG["style"] > 3:
    shape_network(MODEL_CONFIG, FOLDER_RESULTS)

## sceduler

In [ ]:
# from matplotlib import pyplot
# import optax

# init_value = 0.002
# step_2, step_3 = 5000, 10000
# lr2, lr3 = 0.5, 0.1

# piecewise_constant_schedule = optax.schedules.piecewise_constant_schedule(
#     init_value=init_value,
#     boundaries_and_scales={
#         int(step_2): lr2,
#         int(step_3): lr3,
#     },
# )
# transition = 4400
# peak_value = 2.5e-3
# cosine_onecycle_scheduler = optax.schedules.cosine_onecycle_schedule(
#     transition_steps=transition,
#     peak_value=peak_value,
#     pct_start=0.3,
#     div_factor=10.0,
#     final_div_factor=1.4e1,
# )

# boundary = 2500
# last_cosine_lr = [cosine_onecycle_scheduler(step) for step in arange(0, boundary)][-1:][0]
# pids = optax.schedules.piecewise_interpolate_schedule(
#     interpolate_type="cosine",
#     init_value=last_cosine_lr,
#     boundaries_and_scales={
#         int(step_2): lr2,
#         int(step_3): lr3,
#     },
# )

# joined_schedule = optax.schedules.join_schedules(
#     [cosine_onecycle_scheduler, pids],
#     boundaries=[boundary],
# )

# pids2 = optax.schedules.piecewise_interpolate_schedule(
#     interpolate_type="cosine",
#     init_value=0.002,
#     boundaries_and_scales={
#         int(step_2): lr2,
#         int(step_3): lr3,
#     },
# )

# steps = arange(0, 10000)
# piecewise_lr = [piecewise_constant_schedule(step) for step in steps]
# pids2_lr = [pids2(step) for step in steps]
# joined_lr = [joined_schedule(step) for step in steps]

# pyplot.figure(figsize=(10, 6))
# pyplot.plot(steps, piecewise_lr, label="Piecewise Constant Schedule")
# pyplot.plot(steps, pids2_lr, label="pids2_lr")
# pyplot.plot(steps, joined_lr, label="Joined Schedule (Cosine OneCycle + PIDS)")
# pyplot.axvline(boundary, color="yellow", linestyle="--", label=f"Boundary: {boundary}")
# pyplot.xlabel("Steps")
# pyplot.ylabel("Learning Rate")
# pyplot.title("Learning Rate Schedules")
# pyplot.legend()
# pyplot.grid(True)
# pyplot.show()